# 🤖 auto-mat-ion Sensor Analysis

Análisis estadístico completo de las capacidades de sensores del navegador.

## Categorías de Sensores

| Categoría | Sensores | API |
|-----------|----------|-----|
| **Media** | Camera, Microphone, Screen | getUserMedia, getDisplayMedia |
| **Motion** | Accelerometer, Gyroscope, Orientation | Sensor APIs, DeviceMotion |
| **Environment** | Geolocation, Ambient Light, Proximity | Geolocation API, Sensor APIs |
| **Connectivity** | Bluetooth, USB, NFC, Serial | Web Bluetooth, WebUSB, Web NFC |
| **Hardware** | Battery, Vibration, Gamepad, Wake Lock | Navigator APIs |

---

## 1. Setup & Configuration

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json
import os
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", palette="viridis")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Colors matching auto-mat-ion theme
COLORS = {
    'primary': '#00d4aa',
    'secondary': '#7c3aed',
    'success': '#22c55e',
    'warning': '#f59e0b',
    'error': '#ef4444',
    'bg': '#0f0f1a',
    'surface': '#1a1a2e'
}

print("✅ Libraries loaded successfully")
print(f"📊 Pandas version: {pd.__version__}")
print(f"🔢 NumPy version: {np.__version__}")

In [ ]:
# Data paths
ANALYSIS_DIR = Path.cwd()
PROJECT_ROOT = ANALYSIS_DIR.parent
LOGS_DIR = PROJECT_ROOT / 'logs'
DEMO_DIR = PROJECT_ROOT / 'demo'

print(f"📁 Analysis directory: {ANALYSIS_DIR}")
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📁 Logs directory: {LOGS_DIR}")

# Create output directory
OUTPUT_DIR = ANALYSIS_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"📁 Output directory: {OUTPUT_DIR}")

## 2. Data Loading Utilities

In [ ]:
def load_test_results(session_dir: Path) -> pd.DataFrame:
    """
    Load test results from a session directory.
    Supports JSON, CSV, and NDJSON formats.
    """
    results = []
    
    for file in session_dir.glob('*.json'):
        try:
            with open(file, 'r') as f:
                data = json.load(f)
                if isinstance(data, list):
                    results.extend(data)
                else:
                    results.append(data)
        except Exception as e:
            print(f"⚠️ Error loading {file}: {e}")
    
    for file in session_dir.glob('*.csv'):
        try:
            df = pd.read_csv(file)
            results.extend(df.to_dict('records'))
        except Exception as e:
            print(f"⚠️ Error loading {file}: {e}")
    
    return pd.DataFrame(results) if results else pd.DataFrame()


def load_device_capabilities(file_path: Path) -> dict:
    """
    Load device capabilities from a JSON file.
    """
    try:
        with open(file_path, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f"⚠️ Error loading capabilities: {e}")
        return {}


def get_latest_session() -> Path | None:
    """
    Get the most recent session directory.
    """
    if not LOGS_DIR.exists():
        return None
    
    sessions = [d for d in LOGS_DIR.iterdir() if d.is_dir() and d.name.startswith('session_')]
    if not sessions:
        return None
    
    return max(sessions, key=lambda x: x.stat().st_mtime)

print("✅ Data loading utilities defined")

## 3. Sample Data Generation

Generamos datos de ejemplo para demostrar las capacidades de análisis. En producción, estos datos vendrán de los tests ejecutados.

In [ ]:
def generate_sample_video_data(n_samples: int = 100) -> pd.DataFrame:
    """
    Generate sample video sensor test data.
    """
    np.random.seed(42)
    
    resolutions = ['640x480', '1280x720', '1920x1080', '3840x2160']
    facing_modes = ['user', 'environment']
    frame_rates = [15, 24, 30, 60]
    devices = ['Pixel 6', 'iPhone 13', 'Samsung S21', 'Emulator']
    browsers = ['Chrome', 'Firefox', 'Safari']
    
    data = {
        'test_id': range(1, n_samples + 1),
        'timestamp': pd.date_range(start='2026-01-01', periods=n_samples, freq='5min'),
        'device': np.random.choice(devices, n_samples),
        'browser': np.random.choice(browsers, n_samples),
        'resolution': np.random.choice(resolutions, n_samples),
        'facing_mode': np.random.choice(facing_modes, n_samples),
        'target_fps': np.random.choice(frame_rates, n_samples),
        'actual_fps': np.random.normal(30, 5, n_samples).clip(10, 60),
        'latency_ms': np.random.exponential(50, n_samples) + 20,
        'success': np.random.choice([True, False], n_samples, p=[0.92, 0.08]),
        'torch_available': np.random.choice([True, False], n_samples, p=[0.7, 0.3]),
        'autofocus_available': np.random.choice([True, False], n_samples, p=[0.85, 0.15]),
    }
    
    return pd.DataFrame(data)


def generate_sample_audio_data(n_samples: int = 100) -> pd.DataFrame:
    """
    Generate sample audio sensor test data.
    """
    np.random.seed(43)
    
    sample_rates = [8000, 16000, 44100, 48000, 96000]
    channels = [1, 2]
    devices = ['Pixel 6', 'iPhone 13', 'Samsung S21', 'MacBook Pro']
    
    data = {
        'test_id': range(1, n_samples + 1),
        'timestamp': pd.date_range(start='2026-01-01', periods=n_samples, freq='3min'),
        'device': np.random.choice(devices, n_samples),
        'sample_rate': np.random.choice(sample_rates, n_samples),
        'channels': np.random.choice(channels, n_samples),
        'echo_cancellation': np.random.choice([True, False], n_samples, p=[0.8, 0.2]),
        'noise_suppression': np.random.choice([True, False], n_samples, p=[0.75, 0.25]),
        'measured_latency_ms': np.random.exponential(30, n_samples) + 10,
        'rms_level_db': np.random.normal(-20, 5, n_samples),
        'success': np.random.choice([True, False], n_samples, p=[0.95, 0.05]),
    }
    
    return pd.DataFrame(data)


def generate_sample_motion_data(n_samples: int = 500) -> pd.DataFrame:
    """
    Generate sample motion sensor test data.
    """
    np.random.seed(44)
    
    sensors = ['accelerometer', 'gyroscope', 'magnetometer']
    frequencies = [10, 30, 60, 120]
    devices = ['Pixel 6', 'iPhone 13', 'Samsung S21']
    
    data = {
        'test_id': range(1, n_samples + 1),
        'timestamp': pd.date_range(start='2026-01-01', periods=n_samples, freq='1min'),
        'device': np.random.choice(devices, n_samples),
        'sensor': np.random.choice(sensors, n_samples),
        'frequency_hz': np.random.choice(frequencies, n_samples),
        'x': np.random.normal(0, 5, n_samples),
        'y': np.random.normal(0, 5, n_samples),
        'z': np.random.normal(9.8, 0.5, n_samples),  # Gravity on Z
        'sampling_jitter_ms': np.random.exponential(2, n_samples),
        'success': np.random.choice([True, False], n_samples, p=[0.98, 0.02]),
    }
    
    return pd.DataFrame(data)


# Generate sample data
video_df = generate_sample_video_data(100)
audio_df = generate_sample_audio_data(80)
motion_df = generate_sample_motion_data(500)

print(f"✅ Generated sample data:")
print(f"   📹 Video: {len(video_df)} samples")
print(f"   🎤 Audio: {len(audio_df)} samples")
print(f"   📐 Motion: {len(motion_df)} samples")

## 4. Video Sensor Analysis

Análisis de capacidades de cámara: resolución, frame rate, latencia, torch, autofocus.

In [ ]:
# Video test success rate by device
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Success rate by device
success_by_device = video_df.groupby('device')['success'].mean() * 100
ax1 = axes[0]
bars = ax1.bar(success_by_device.index, success_by_device.values, color=COLORS['primary'])
ax1.set_ylabel('Success Rate (%)')
ax1.set_title('📷 Camera Test Success Rate by Device')
ax1.set_ylim(0, 100)
ax1.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, success_by_device.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', 
             ha='center', va='bottom', fontsize=9)

# FPS distribution
ax2 = axes[1]
ax2.hist(video_df['actual_fps'], bins=20, color=COLORS['secondary'], edgecolor='white', alpha=0.8)
ax2.axvline(video_df['actual_fps'].mean(), color=COLORS['warning'], linestyle='--', label=f'Mean: {video_df["actual_fps"].mean():.1f} fps')
ax2.set_xlabel('Actual FPS')
ax2.set_ylabel('Count')
ax2.set_title('📊 Frame Rate Distribution')
ax2.legend()

# Latency by resolution
ax3 = axes[2]
resolution_order = ['640x480', '1280x720', '1920x1080', '3840x2160']
video_df['resolution'] = pd.Categorical(video_df['resolution'], categories=resolution_order, ordered=True)
sns.boxplot(data=video_df, x='resolution', y='latency_ms', ax=ax3, palette='viridis')
ax3.set_xlabel('Resolution')
ax3.set_ylabel('Latency (ms)')
ax3.set_title('⏱️ Capture Latency by Resolution')
ax3.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'video_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Video Analysis Summary:")
print(f"   Overall success rate: {video_df['success'].mean()*100:.1f}%")
print(f"   Mean FPS: {video_df['actual_fps'].mean():.1f} ± {video_df['actual_fps'].std():.1f}")
print(f"   Mean latency: {video_df['latency_ms'].mean():.1f} ± {video_df['latency_ms'].std():.1f} ms")
print(f"   Torch available: {video_df['torch_available'].mean()*100:.1f}%")
print(f"   Autofocus available: {video_df['autofocus_available'].mean()*100:.1f}%")

## 5. Audio Sensor Analysis

Análisis de capacidades de micrófono: sample rate, canales, procesamiento de audio.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Success rate by sample rate
ax1 = axes[0]
sr_success = audio_df.groupby('sample_rate')['success'].mean() * 100
ax1.bar([str(sr) for sr in sr_success.index], sr_success.values, color=COLORS['primary'])
ax1.set_xlabel('Sample Rate (Hz)')
ax1.set_ylabel('Success Rate (%)')
ax1.set_title('🎤 Audio Success Rate by Sample Rate')
ax1.set_ylim(0, 100)

# Latency distribution
ax2 = axes[1]
ax2.hist(audio_df['measured_latency_ms'], bins=20, color=COLORS['secondary'], edgecolor='white', alpha=0.8)
median_lat = audio_df['measured_latency_ms'].median()
ax2.axvline(median_lat, color=COLORS['error'], linestyle='--', label=f'Median: {median_lat:.1f} ms')
ax2.set_xlabel('Latency (ms)')
ax2.set_ylabel('Count')
ax2.set_title('⏱️ Audio Latency Distribution')
ax2.legend()

# Audio processing features
ax3 = axes[2]
features = ['echo_cancellation', 'noise_suppression']
feature_rates = [audio_df[f].mean() * 100 for f in features]
bars = ax3.bar(['Echo\nCancellation', 'Noise\nSuppression'], feature_rates, color=[COLORS['success'], COLORS['warning']])
ax3.set_ylabel('Availability (%)')
ax3.set_title('🔊 Audio Processing Features')
ax3.set_ylim(0, 100)
for bar, val in zip(bars, feature_rates):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', 
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'audio_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n🎤 Audio Analysis Summary:")
print(f"   Overall success rate: {audio_df['success'].mean()*100:.1f}%")
print(f"   Median latency: {audio_df['measured_latency_ms'].median():.1f} ms")
print(f"   Mean RMS level: {audio_df['rms_level_db'].mean():.1f} dB")

## 6. Motion Sensor Analysis

Análisis de acelerómetro, giroscopio, magnetómetro.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Success rate by sensor type
ax1 = axes[0, 0]
sensor_success = motion_df.groupby('sensor')['success'].mean() * 100
colors_list = [COLORS['primary'], COLORS['secondary'], COLORS['warning']]
bars = ax1.bar(sensor_success.index, sensor_success.values, color=colors_list)
ax1.set_ylabel('Success Rate (%)')
ax1.set_title('📐 Motion Sensor Success Rate')
ax1.set_ylim(0, 100)

# Sampling jitter by frequency
ax2 = axes[0, 1]
sns.boxplot(data=motion_df, x='frequency_hz', y='sampling_jitter_ms', ax=ax2, palette='coolwarm')
ax2.set_xlabel('Sampling Frequency (Hz)')
ax2.set_ylabel('Jitter (ms)')
ax2.set_title('📉 Sampling Jitter by Frequency')

# Accelerometer readings distribution
ax3 = axes[1, 0]
accel_data = motion_df[motion_df['sensor'] == 'accelerometer']
ax3.scatter(accel_data['x'], accel_data['y'], c=accel_data['z'], cmap='viridis', alpha=0.6, s=20)
ax3.set_xlabel('X axis (m/s²)')
ax3.set_ylabel('Y axis (m/s²)')
ax3.set_title('🧭 Accelerometer XY Distribution (color=Z)')
cbar = plt.colorbar(ax3.collections[0], ax=ax3)
cbar.set_label('Z axis (m/s²)')

# Device comparison
ax4 = axes[1, 1]
device_stats = motion_df.groupby(['device', 'sensor']).size().unstack(fill_value=0)
device_stats.plot(kind='bar', ax=ax4, colormap='viridis')
ax4.set_xlabel('Device')
ax4.set_ylabel('Number of Tests')
ax4.set_title('📱 Test Distribution by Device & Sensor')
ax4.tick_params(axis='x', rotation=45)
ax4.legend(title='Sensor')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'motion_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📐 Motion Sensor Analysis Summary:")
print(f"   Overall success rate: {motion_df['success'].mean()*100:.1f}%")
print(f"   Mean jitter: {motion_df['sampling_jitter_ms'].mean():.2f} ms")
print(f"   Sensors tested: {motion_df['sensor'].nunique()}")

## 7. Cross-Sensor Correlation Analysis

In [ ]:
# Create device summary across all sensors
device_summary = pd.DataFrame({
    'Device': video_df['device'].unique()
})

# Video metrics
video_summary = video_df.groupby('device').agg({
    'success': 'mean',
    'actual_fps': 'mean',
    'latency_ms': 'mean',
}).reset_index()
video_summary.columns = ['Device', 'Video_Success', 'Video_FPS', 'Video_Latency']

# Audio metrics
audio_summary = audio_df.groupby('device').agg({
    'success': 'mean',
    'measured_latency_ms': 'mean',
}).reset_index()
audio_summary.columns = ['Device', 'Audio_Success', 'Audio_Latency']

# Motion metrics
motion_summary = motion_df.groupby('device').agg({
    'success': 'mean',
    'sampling_jitter_ms': 'mean',
}).reset_index()
motion_summary.columns = ['Device', 'Motion_Success', 'Motion_Jitter']

# Merge all
all_metrics = video_summary.merge(audio_summary, on='Device', how='outer')
all_metrics = all_metrics.merge(motion_summary, on='Device', how='outer')

print("📊 Cross-Sensor Device Summary:")
print("="*80)
display(all_metrics.round(2))

In [ ]:
# Correlation heatmap
numeric_cols = all_metrics.select_dtypes(include=[np.number]).columns
correlation = all_metrics[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, ax=ax, linewidths=0.5)
ax.set_title('🔗 Cross-Sensor Metric Correlations')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Statistical Tests

In [ ]:
from scipy.stats import ttest_ind, mannwhitneyu, kruskal

print("📈 Statistical Analysis")
print("="*80)

# Test 1: FPS difference between facing modes
front_fps = video_df[video_df['facing_mode'] == 'user']['actual_fps']
back_fps = video_df[video_df['facing_mode'] == 'environment']['actual_fps']

stat, p_value = ttest_ind(front_fps, back_fps)
print(f"\n1. FPS Difference (Front vs Back Camera):")
print(f"   Front mean: {front_fps.mean():.2f} fps")
print(f"   Back mean: {back_fps.mean():.2f} fps")
print(f"   t-statistic: {stat:.3f}, p-value: {p_value:.4f}")
print(f"   Significant difference: {'Yes' if p_value < 0.05 else 'No'}")

# Test 2: Latency difference across resolutions
groups = [group['latency_ms'].values for name, group in video_df.groupby('resolution')]
stat, p_value = kruskal(*groups)
print(f"\n2. Latency Difference Across Resolutions (Kruskal-Wallis):")
print(f"   H-statistic: {stat:.3f}, p-value: {p_value:.4f}")
print(f"   Significant difference: {'Yes' if p_value < 0.05 else 'No'}")

# Test 3: Motion sensor jitter by frequency
low_freq = motion_df[motion_df['frequency_hz'] <= 30]['sampling_jitter_ms']
high_freq = motion_df[motion_df['frequency_hz'] > 30]['sampling_jitter_ms']

stat, p_value = mannwhitneyu(low_freq, high_freq)
print(f"\n3. Jitter Difference (Low vs High Frequency):")
print(f"   Low freq (≤30Hz) median: {low_freq.median():.2f} ms")
print(f"   High freq (>30Hz) median: {high_freq.median():.2f} ms")
print(f"   U-statistic: {stat:.3f}, p-value: {p_value:.4f}")
print(f"   Significant difference: {'Yes' if p_value < 0.05 else 'No'}")

## 9. Sensor Capability Matrix

In [ ]:
# Define sensor capability matrix
SENSOR_CAPABILITIES = {
    'Media': {
        'Camera': {'api': 'getUserMedia(video)', 'status': '✅', 'coverage': 95},
        'Microphone': {'api': 'getUserMedia(audio)', 'status': '✅', 'coverage': 98},
        'Screen Capture': {'api': 'getDisplayMedia()', 'status': '✅', 'coverage': 85},
        'MediaDevices': {'api': 'enumerateDevices()', 'status': '✅', 'coverage': 99},
    },
    'Motion': {
        'Accelerometer': {'api': 'Accelerometer API', 'status': '✅', 'coverage': 92},
        'Gyroscope': {'api': 'Gyroscope API', 'status': '✅', 'coverage': 90},
        'Magnetometer': {'api': 'Magnetometer API', 'status': '⚠️', 'coverage': 65},
        'DeviceOrientation': {'api': 'DeviceOrientationEvent', 'status': '✅', 'coverage': 95},
        'DeviceMotion': {'api': 'DeviceMotionEvent', 'status': '✅', 'coverage': 95},
    },
    'Environment': {
        'Geolocation': {'api': 'Geolocation API', 'status': '✅', 'coverage': 99},
        'Ambient Light': {'api': 'AmbientLightSensor', 'status': '⚠️', 'coverage': 40},
        'Proximity': {'api': 'ProximitySensor', 'status': '❌', 'coverage': 15},
    },
    'Connectivity': {
        'Bluetooth': {'api': 'Web Bluetooth API', 'status': '⚠️', 'coverage': 70},
        'USB': {'api': 'WebUSB API', 'status': '⚠️', 'coverage': 60},
        'NFC': {'api': 'Web NFC API', 'status': '⚠️', 'coverage': 35},
        'Serial': {'api': 'Web Serial API', 'status': '⚠️', 'coverage': 55},
    },
    'Hardware': {
        'Battery': {'api': 'Battery Status API', 'status': '⚠️', 'coverage': 75},
        'Vibration': {'api': 'Vibration API', 'status': '✅', 'coverage': 90},
        'Gamepad': {'api': 'Gamepad API', 'status': '✅', 'coverage': 85},
        'Wake Lock': {'api': 'Screen Wake Lock API', 'status': '✅', 'coverage': 80},
    },
}

# Create summary table
rows = []
for category, sensors in SENSOR_CAPABILITIES.items():
    for sensor, info in sensors.items():
        rows.append({
            'Category': category,
            'Sensor': sensor,
            'API': info['api'],
            'Status': info['status'],
            'Browser Coverage (%)': info['coverage']
        })

capability_df = pd.DataFrame(rows)
print("📋 Sensor Capability Matrix:")
print("="*80)
display(capability_df)

In [ ]:
# Visualize capability coverage
fig, ax = plt.subplots(figsize=(14, 8))

# Create grouped bar chart
categories = capability_df['Category'].unique()
x = np.arange(len(capability_df))

# Color by status
colors = capability_df['Status'].map({'✅': COLORS['success'], '⚠️': COLORS['warning'], '❌': COLORS['error']})

bars = ax.barh(capability_df['Sensor'], capability_df['Browser Coverage (%)'], color=colors)
ax.set_xlabel('Browser Coverage (%)')
ax.set_title('🌐 Sensor API Browser Coverage')
ax.set_xlim(0, 105)

# Add coverage values
for bar, val in zip(bars, capability_df['Browser Coverage (%)']):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2, f'{val}%', 
            va='center', fontsize=9)

# Add category labels
current_y = 0
for category in categories:
    count = len(capability_df[capability_df['Category'] == category])
    ax.axhline(y=current_y - 0.5, color='white', linestyle='-', alpha=0.3, linewidth=0.5)
    current_y += count

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sensor_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Export Results

In [ ]:
# Export all data to Excel
with pd.ExcelWriter(OUTPUT_DIR / 'sensor_analysis_report.xlsx', engine='openpyxl') as writer:
    video_df.to_excel(writer, sheet_name='Video Tests', index=False)
    audio_df.to_excel(writer, sheet_name='Audio Tests', index=False)
    motion_df.to_excel(writer, sheet_name='Motion Tests', index=False)
    all_metrics.to_excel(writer, sheet_name='Device Summary', index=False)
    capability_df.to_excel(writer, sheet_name='Capability Matrix', index=False)

print(f"✅ Report exported to: {OUTPUT_DIR / 'sensor_analysis_report.xlsx'}")

# Export summary JSON
summary = {
    'generated_at': datetime.now().isoformat(),
    'video': {
        'total_tests': len(video_df),
        'success_rate': float(video_df['success'].mean()),
        'mean_fps': float(video_df['actual_fps'].mean()),
        'mean_latency_ms': float(video_df['latency_ms'].mean()),
    },
    'audio': {
        'total_tests': len(audio_df),
        'success_rate': float(audio_df['success'].mean()),
        'mean_latency_ms': float(audio_df['measured_latency_ms'].mean()),
    },
    'motion': {
        'total_tests': len(motion_df),
        'success_rate': float(motion_df['success'].mean()),
        'mean_jitter_ms': float(motion_df['sampling_jitter_ms'].mean()),
    },
    'sensors_tested': list(SENSOR_CAPABILITIES.keys()),
}

with open(OUTPUT_DIR / 'analysis_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"✅ Summary exported to: {OUTPUT_DIR / 'analysis_summary.json'}")

---

## 📊 Analysis Complete

Los archivos generados se encuentran en el directorio `output/`:

- `video_analysis.png` - Gráficos de análisis de cámara
- `audio_analysis.png` - Gráficos de análisis de micrófono
- `motion_analysis.png` - Gráficos de análisis de sensores de movimiento
- `correlation_heatmap.png` - Correlaciones entre métricas
- `sensor_coverage.png` - Cobertura de APIs en navegadores
- `sensor_analysis_report.xlsx` - Datos completos en Excel
- `analysis_summary.json` - Resumen en formato JSON

---

*auto-mat-ion Sensor Analysis v1.0.0*